# Agentes para consultas de normativa de pregrado
**Entregable 3 · Ingeniería UdeC · Álvaro Contreras y Pablo Cortés**

Planificación, recuperación de evidencia, generación de respuestas y revisión con Qwen3-4B.

La demostración combina una consulta sobre reglas de asistencia y excepciones (**H17**), un fallo documentado de recuperación y cálculo (**H03**) y la evaluación del sistema sobre **75 consultas**.


## A1 · Entorno de ejecución
Bibliotecas para inferencia, cuantización NF4 y recuperación semántica. La configuración y las versiones utilizadas se registran con cada ejecución.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.51,<5', 'accelerate>=1,<2', 'bitsandbytes>=0.45,<1',
    'sentence-transformers>=4,<6'])
print('Bibliotecas del entorno instaladas.')


## A2 · Código y corpus
Código, prompts y documentos normativos de una versión fija del repositorio. El paquete se valida mediante SHA-256 antes de cargarlo.


In [ ]:
from pathlib import Path
import sys, json, hashlib, tempfile, zipfile, urllib.request, stat
import torch
assert torch.cuda.is_available(), 'GPU CUDA no disponible; esta ejecución requiere GPU.'
print('GPU:', torch.cuda.get_device_name(0))
DESCARGAR_DEL_REPO = True
FUENTE_REF = '753eef66f3f5003646aa2dc401281d1bbac47c68'  # versión comprobada; no depende de cambios en main
REPO = 'Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1'

def leer_url(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'D3-video-Colab'})
    with urllib.request.urlopen(req, timeout=180) as respuesta:
        return respuesta.read()

commit_fuentes = None
if DESCARGAR_DEL_REPO:
    commit_fuentes = FUENTE_REF if len(FUENTE_REF) == 40 and all(c in '0123456789abcdef' for c in FUENTE_REF.lower()) else json.loads(leer_url(f'https://api.github.com/repos/{REPO}/commits/{FUENTE_REF}'))['sha']
    raw = f'https://raw.githubusercontent.com/{REPO}/{commit_fuentes}/Deliverable%203/'
    datos_zip = leer_url(raw + 'Deliverable3_ejecucion.zip')
    manifiesto = json.loads(leer_url(raw + 'auditoria/manifiesto.json'))
    assert hashlib.sha256(datos_zip).hexdigest() == manifiesto['Deliverable3_ejecucion.zip'], 'Hash del paquete incorrecto.'
else:
    from google.colab import files
    subidos = files.upload()
    assert len(subidos) == 1, 'Se requiere un único paquete Deliverable3_ejecucion.zip.'
    nombre, datos_zip = next(iter(subidos.items()))
    assert nombre.endswith('.zip'), 'El archivo debe ser ZIP.'

paquete_sha256 = hashlib.sha256(datos_zip).hexdigest()
raiz = Path(tempfile.mkdtemp(prefix='d3_video_', dir='/content'))
paquete = raiz/'paquete.zip'
paquete.write_bytes(datos_zip)
with zipfile.ZipFile(paquete) as z:
    for item in z.infolist():
        destino = (raiz/item.filename).resolve()
        assert destino.is_relative_to(raiz.resolve()), 'Ruta no válida en ZIP.'
        assert not stat.S_ISLNK(item.external_attr >> 16), 'No se admiten enlaces simbólicos.'
    z.extractall(raiz)
BASE = raiz/'Deliverable3'
assert (BASE/'video/soporte_video.py').is_file(), 'Paquete incompatible: falta el módulo de soporte del video.'
AGENTES = BASE/'agentes'
for ruta in (AGENTES, BASE/'video'):
    sys.path.insert(0, str(ruta))
for modulo in ('orquestador','tools','soporte_video'):
    sys.modules.pop(modulo, None)
print('Fuentes:', BASE)
print('Commit:', commit_fuentes or 'archivo subido; se registra su hash')
print('SHA-256 del ZIP:', paquete_sha256)


## A3 · Modelo generativo
**Qwen3-4B**, compartido por todos los roles, con cuantización NF4. Se conservan los parámetros y presupuestos de generación del sistema evaluado.


In [ ]:
import hashlib
from huggingface_hub import model_info
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODELO = 'Qwen/Qwen3-4B'
REVISION_MODELO = model_info(MODELO).sha
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(MODELO, revision=REVISION_MODELO)
model = AutoModelForCausalLM.from_pretrained(MODELO, revision=REVISION_MODELO,
    quantization_config=bnb, device_map='auto')
model.eval()
MAX_TOKENS = {'enrutador':256, 'recuperador':160, 'redactor':1200, 'verificador':700}
SEMILLA = 2026
PARAMETROS = dict(do_sample=True, temperature=0.7, top_p=0.8, top_k=20, min_p=0.0)

def generar_real(agente, prompt):
    texto = tok.apply_chat_template([{'role':'user','content':prompt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)
    semilla = SEMILLA + int(hashlib.sha256((agente+prompt).encode()).hexdigest()[:8],16) % 1_000_000
    torch.manual_seed(semilla)
    entradas = tok(texto, return_tensors='pt').to(model.device)
    with torch.no_grad():
        salida = model.generate(**entradas, max_new_tokens=MAX_TOKENS[agente],
            pad_token_id=tok.eos_token_id, **PARAMETROS)
    return tok.decode(salida[0][entradas.input_ids.shape[1]:], skip_special_tokens=True).strip()

print('Modelo listo:', MODELO, '| revisión:', REVISION_MODELO, '| cómputo:', DTYPE)


## A4 · Recuperación de evidencia
**Multilingual E5-small** representa las consultas y los fragmentos del corpus. La herramienta de búsqueda selecciona las unidades normativas por similitud semántica.


In [ ]:
from tools import Corpus
from sentence_transformers import SentenceTransformer
import numpy as np
MODELO_E5 = 'intfloat/multilingual-e5-small'
REVISION_E5 = model_info(MODELO_E5).sha
corpus = Corpus(AGENTES)
e5 = SentenceTransformer(MODELO_E5, revision=REVISION_E5, device='cuda')
fragmentos = corpus.frag
vectores = e5.encode([f['texto_embedding'] for f in fragmentos],
    normalize_embeddings=True, batch_size=64, show_progress_bar=True)

def buscar_e5(consulta, k=5):
    q = e5.encode(['query: '+str(consulta)], normalize_embeddings=True)[0]
    orden = np.argsort(-(vectores @ q))
    vistos, encontrados = set(), []
    for i in orden:
        f = fragmentos[int(i)]
        uid = f.get('unidad_id') or f.get('parent_id')
        if uid in vistos:
            continue
        vistos.add(uid)
        u = corpus.por_id.get(uid, {})
        encontrados.append({'id':uid, 'texto':u.get('texto', f['texto'])})
        if len(encontrados) >= k:
            break
    return encontrados

corpus.buscar = buscar_e5
print('E5 listo:', len(corpus.por_id), 'unidades y', len(fragmentos), 'fragmentos.')


## A5 · Configuración y trazabilidad
Cada consulta conserva el entorno, los prompts completos, las salidas del modelo, los resultados de las herramientas y las decisiones del controlador.


In [ ]:
import csv, platform, importlib.metadata
from soporte_video import (SesionDemo, mostrar_portada, mostrar_fallo,
                           mostrar_resultados, exportar_evidencia)
versiones = {p:importlib.metadata.version(p) for p in
    ('torch','transformers','accelerate','bitsandbytes','sentence-transformers',
     'huggingface-hub','tokenizers','numpy','safetensors')}
assert importlib.metadata.version('transformers') == __import__('transformers').__version__, 'La versión importada de Transformers difiere de la instalada.'
ENTORNO = {'python':platform.python_version(), 'gpu':torch.cuda.get_device_name(0),
    'cuda':torch.version.cuda, 'versiones':versiones, 'modelo':MODELO,
    'revision_modelo':REVISION_MODELO, 'embeddings':MODELO_E5, 'revision_e5':REVISION_E5,
    'cuantizacion':'NF4 con doble cuantización', 'dtype':str(DTYPE),
    'parametros':PARAMETROS, 'max_tokens':MAX_TOKENS, 'enable_thinking':False,
    'semilla_base':SEMILLA, 'regla_semilla':'base + int(sha256(agente+prompt)[:8],16) % 1000000',
    'commit_fuentes':commit_fuentes, 'paquete_sha256':paquete_sha256,
    'fuentes_sha256':{str(p.relative_to(BASE)):hashlib.sha256(p.read_bytes()).hexdigest()
       for carpeta in ('agentes','prompts','video') for p in sorted((BASE/carpeta).rglob('*'))
       if p.is_file() and '__pycache__' not in p.parts and p.suffix in ('.py','.txt','.json','.csv')}}
with (AGENTES/'test_set_dificil_25.csv').open(encoding='utf-8-sig', newline='') as f:
    preguntas = {str(r['id']):r['pregunta'] for r in csv.DictReader(f)}
assert 'H17' in preguntas and 'H03' in preguntas
assert (BASE/'documento/grafo_compacto.png').is_file(), 'El paquete debe incluir el grafo actualizado.'
ENTORNO['grafo_sha256'] = hashlib.sha256((BASE/'documento/grafo_compacto.png').read_bytes()).hexdigest()
EVIDENCIA = Path('/content/D3_video_evidencia')
EVIDENCIA.mkdir(exist_ok=True)
sesiones_video = []
ultima_por_caso = {}
print('Configuración registrada. Modelo:', MODELO)
print('Consulta H17:', preguntas['H17'])


# Demostración del sistema
## B0 · Arquitectura y roles
Las decisiones del Enrutador, Recuperador y Verificador determinan las búsquedas, correcciones y condiciones de parada.


In [ ]:
mostrar_portada(BASE)


## B1 · Reglas de asistencia y excepciones
**H17 · Ejecución real**

> Explica los requisitos de asistencia que puede exigir un profesor en la Facultad de Ingeniería y en qué casos ese tope puede ser distinto.

La consulta exige distinguir la regla general, las actividades con requisitos distintos y las excepciones respaldadas por la normativa.


In [ ]:
sesion_h17 = SesionDemo(corpus, generar_real, EVIDENCIA, ENTORNO,
    caso='H17', pregunta=preguntas['H17'])
sesiones_video.append(sesion_h17)
ultima_por_caso['H17'] = sesion_h17.id
resultado_h17 = sesion_h17.ejecutar()


## B2 · Fallo de recuperación y cálculo
**H03 · Registro histórico**

La pregunta requiere recuperar cuatro fechas del calendario, calcular los plazos de ambos semestres y determinar cuál es mayor. Se comparan el primer borrador, las correcciones y la respuesta final de la ejecución registrada.


In [ ]:
fallo_historico = mostrar_fallo(BASE)


## B3 · Evaluación y alcance
Resultados del conjunto original de **50 consultas** y del conjunto de **25 consultas difíciles**, bajo el mismo criterio de corrección. La evaluación global es independiente de la consulta ejecutada en esta demostración.


In [ ]:
mostrar_resultados()


# Evidencia de la ejecución
## C1 · Paquete de trazabilidad
Prompts, respuestas, herramientas, entorno y fuentes de cada intento. El índice identifica la ejecución seleccionada; H03 se conserva por separado como caso histórico.


In [ ]:
RUN_IDS_EN_VIDEO = [ultima_por_caso[c] for c in ('H17',) if c in ultima_por_caso]
for s in sesiones_video:
    print('EN VIDEO' if s.id in RUN_IDS_EN_VIDEO else 'otro intento',
          s.id, s.datos['caso'], s.datos['estado'], s.datos['pregunta'])
paquete_evidencia = exportar_evidencia(BASE, EVIDENCIA, sesiones_video, RUN_IDS_EN_VIDEO, ENTORNO)
print('Evidencia guardada:', paquete_evidencia)
from google.colab import files
files.download(str(paquete_evidencia))


## C2 · Mensajes completos
Consulta del prompt y la salida cruda de una llamada del modelo. La vista de presentación y el registro descargable corresponden a la misma ejecución.


In [ ]:
SESION_A_REVISAR = sesiones_video[-1]
NUMERO_LLAMADA = 1
llamada = SESION_A_REVISAR.datos['llamadas'][NUMERO_LLAMADA-1]
print('AGENTE:', llamada['agente'])
print('PROMPT COMPLETO:\n', llamada['mensajes'][0]['content'])
print('\nSALIDA CRUDA:\n', llamada.get('salida_cruda', '(llamada incompleta)'))


## Fuentes técnicas
- [Qwen3-4B](https://huggingface.co/Qwen/Qwen3-4B).
- [Cuantización NF4 con bitsandbytes](https://huggingface.co/docs/transformers/quantization/bitsandbytes).
- [Repositorio del entregable](https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1/tree/main/Deliverable%203).
